# Structure report

Report generator with the same visual layout and plotting style as the reference script.

In [1]:
# User settings

from pathlib import Path

TRAJ_DIR = Path("tutorial_structures")
OUTPUT_DIR = Path("reports")

MY_LATTICE = 3.61
MY_BIN_SIZE_LATTICE = 0.035
CUTOFF = 0.85 * MY_LATTICE

upload_snapshot = False

# Used only when upload_snapshot = True.
SNAPSHOT_DIR = TRAJ_DIR
SNAPSHOT_SUFFIX = ".png"

SNAPSHOT_WIDTH = 0.35
SNAPSHOT_HEIGHT = 0.50
PDDF_SNAPSHOT_CENTER_X = 0.78
PDDF_SNAPSHOT_CENTER_Y = 0.75

# Used only if an aGCN PNG already exists.
AGCN_SNAPSHOT_CENTER_X = 0.23
AGCN_SNAPSHOT_CENTER_Y = 0.75

MAX_CNAPS_TO_DISPLAY = 10
CNAP_FONT_SIZE = 7.5
CNAP_MAX_CHARS_PER_LINE = 32
CNAP_TABLE_TOP = 0.525
CNAP_TABLE_BOTTOM = -0.60
CNAP_LINE_SPACING = 1.05
CNAP_ENTRY_GAP_UNITS = 0.72
CNAP_NUMBER_X = 0.012
CNAP_TEXT_X = 0.070
CNAP_NATOMS_X = 0.985

MAX_BARS_WITH_VALUE_LABELS = 10

AGCN_MIN = 2.0
AGCN_MAX = 12.0
AGCN_BIN_WIDTH = 0.25
COLOR_SCALE_MIN = 0.0
COLOR_SCALE_MAX = 12.0

TARGET_CNA_LABEL = "(4,2,1)"
CNA_RIGHT_YMAX = 100.0
AGCN_RIGHT_YMAX = 60.0

SAVE_AGCN_XYZ = False
SAVE_PDF = False
OUTPUT_PDF = OUTPUT_DIR / "Full_report.pdf"

# Horizontal B5-style figure.
FIGURE_WIDTH_CM = 16.0
FIGURE_HEIGHT_CM = 8.6
OUTPUT_DPI = 600


In [2]:
# Imports and plotting style

from collections import Counter


import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import Normalize
from matplotlib.cm import get_cmap

from ase import Atoms
from ase.io import read
from ase.visualize.plot import plot_atoms

from snow.io.xyz import read_xyz, write_xyz
from snow.descriptors import cna
from snow.descriptors.distributions import pddf_calculator, com_rdf_calculator
from snow.descriptors.coordination import agcn_calculator
from snow.descriptors.shape_descriptors import (
    gyr_desc,
    aspect_ratio,
    gyr_rad,
    branch_factor,
)

try:
    from snow.descriptors.shape import eccentricity
except ImportError:
    from snow.descriptors.shape_descriptors import eccentricity

MY_BIN_SIZE_AA = MY_BIN_SIZE_LATTICE * MY_LATTICE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CM_TO_INCH = 1.0 / 2.54
FIGURE_SIZE = (
    FIGURE_WIDTH_CM * CM_TO_INCH,
    FIGURE_HEIGHT_CM * CM_TO_INCH,
)

MAIN_COLOR = "#A94E45"
CNA_COLOR = MAIN_COLOR
CNA_TARGET_COLOR = "#D98A80"
EDGE_COLOR = "#303030"
TEXT_COLOR = "#222222"
LIGHT_GREY = "#CFCFCF"
VERY_LIGHT_GREY = "#E8E8E8"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIX Two Text", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 9.0,
    "axes.labelsize": 8.8,
    "axes.titlesize": 9.0,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8.0,
    "axes.linewidth": 0.85,
    "axes.labelcolor": TEXT_COLOR,
    "axes.edgecolor": EDGE_COLOR,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 3.2,
    "ytick.major.size": 3.2,
    "xtick.major.width": 0.80,
    "ytick.major.width": 0.80,
    "xtick.color": EDGE_COLOR,
    "ytick.color": EDGE_COLOR,
    "lines.linewidth": 1.2,
    "figure.facecolor": "white",
    "savefig.dpi": OUTPUT_DPI,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.025,
    "savefig.facecolor": "white",
})


In [ ]:
# ============================================================
# Helper functions
# ============================================================

def count_cna_patterns(cna_atom):
    pattern_counter = Counter()

    for sigs, counts in cna_atom:
        expanded = []
        for sig, cnt in zip(sigs, counts):
            expanded.extend([tuple(sig)] * cnt)
        pattern_counter[tuple(sorted(expanded))] += 1

    return len(cna_atom), len(pattern_counter), pattern_counter


def format_pattern_lines(pattern, max_chars_per_line=32):
    sig_counts = Counter(pattern)

    parts = [
        f"{n}x[{int(r)},{int(s)},{int(t)}]"
        for (r, s, t), n in sorted(sig_counts.items())
    ]

    lines = []
    current_line = ""

    for part in parts:
        candidate = (
            part
            if not current_line
            else current_line + ", " + part
        )

        if len(candidate) <= max_chars_per_line:
            current_line = candidate
        else:
            if current_line:
                lines.append(current_line)
            current_line = part

    if current_line:
        lines.append(current_line)

    return "\n".join(lines)


def padded_ylim(values, pad=1.15, fallback=1.0):
    values = np.asarray(values, dtype=float)

    if values.size == 0:
        return fallback

    vmax = np.nanmax(values)

    if not np.isfinite(vmax) or vmax <= 0:
        return fallback

    return vmax * pad


def clean_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        axis="both",
        direction="out",
        length=3.2,
        width=0.80,
        pad=2.5,
    )

    ax.grid(False)


def add_panel_label(ax, label, x=-0.16, y=1.04):
    ax.text(
        x,
        y,
        label,
        transform=ax.transAxes,
        fontsize=11.0,
        fontweight="bold",
        ha="left",
        va="bottom",
        clip_on=False,
    )


# ============================================================
# Generic image snapshot
# ============================================================

def add_snapshot(
    ax,
    image_path,
    center_x,
    center_y,
    width,
    height,
):
    if image_path is None or not image_path.exists():
        return

    try:
        image = plt.imread(image_path)

        left = center_x - width / 2.0
        bottom = center_y - height / 2.0

        inset = ax.inset_axes(
            [left, bottom, width, height],
            zorder=6,
        )

        inset.imshow(
            image,
            interpolation="lanczos",
        )

        inset.axis("off")

    except Exception as exc:
        print(
            f"WARNING: could not load "
            f"{image_path}: {exc}"
        )


# ============================================================
# Shape table
# ============================================================

def draw_shape_table(ax, shape_rows):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    row_height = 1.0 / len(shape_rows)

    for i, (label, value) in enumerate(shape_rows):

        y_bottom = 1.0 - (i + 1) * row_height

        facecolor = (
            "0.975"
            if i % 2 == 0
            else "1.0"
        )

        rect = Rectangle(
            (0.0, y_bottom),
            1.0,
            row_height,
            transform=ax.transAxes,
            facecolor=facecolor,
            edgecolor=LIGHT_GREY,
            linewidth=0.70,
            clip_on=False,
        )

        ax.add_patch(rect)

        y_center = (
            y_bottom
            + 0.50 * row_height
        )

        ax.text(
            0.50,
            y_center + 0.13 * row_height,
            label,
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=8.3,
            color="0.32",
        )

        ax.text(
            0.50,
            y_center - 0.14 * row_height,
            value,
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=9.7,
            fontweight="semibold",
            color=TEXT_COLOR,
        )


# ============================================================
# Standard ASE structure snapshot
# ============================================================

def make_structure_snapshot(
    struct_path,
    el,
    coords,
    output_path,
):
    atoms = Atoms(
        symbols=list(el),
        positions=np.asarray(coords),
    )

    fig, ax = plt.subplots(
        figsize=(4, 4),
    )

    plot_atoms(
        atoms,
        ax=ax,
        radii=0.42,       # più grosso rispetto a 0.35
    )

    ax.set_axis_off()

    fig.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.02,
        facecolor="white",
    )

    plt.close(fig)


def add_structure_snapshot(
    ax,
    el,
    coords,
    center_x,
    center_y,
    width,
    height,
):
    """
    Draw a standard ASE structure snapshot directly
    inside an existing matplotlib axis.
    """

    atoms = Atoms(
        symbols=list(el),
        positions=np.asarray(coords),
    )

    inset = ax.inset_axes(
        [
            center_x - width / 2.0,
            center_y - height / 2.0,
            width,
            height,
        ],
        zorder=6,
    )

    plot_atoms(
        atoms,
        ax=inset,
        radii=0.42,       # più grosso
    )

    for artist in inset.patches:
        artist.set_linewidth(0.15)

    inset.set_axis_off()


# ============================================================
# aGCN ASE snapshot
# ============================================================

def add_agcn_snapshot(
    ax,
    el,
    coords,
    agcns,
    center_x,
    center_y,
    width,
    height,
):
    """
    Draw an ASE snapshot coloured according to aGCN values.

    IMPORTANT:
    - one colour per atom;
    - same rainbow colormap used by the histogram;
    - same COLOR_SCALE_MIN / COLOR_SCALE_MAX;
    - same global norm;
    """

    coords = np.asarray(coords)
    agcns = np.asarray(agcns, dtype=float)

    if len(coords) != len(agcns):
        raise ValueError(
            "Number of aGCN values does not match "
            "number of atoms: "
            f"{len(agcns)} vs {len(coords)}"
        )

    atoms = Atoms(
        symbols=list(el),
        positions=coords,
    )

    inset = ax.inset_axes(
        [
            center_x - width / 2.0,
            center_y - height / 2.0,
            width,
            height,
        ],
        zorder=6,
    )

    atom_colors = cmap(norm(agcns))

    plot_atoms(
        atoms,
        ax=inset,
        radii=0.42,       # atomi leggermente più grandi
        colors=atom_colors,
    )

    for artist in inset.patches:
        artist.set_linewidth(0.15)

    inset.set_axis_off()

In [ ]:
# ============================================================
# Generate reports
# ============================================================

# structures = sorted(TRAJ_DIR.glob("*.xyz"))

structures = sorted(
    TRAJ_DIR.glob("*h.xyz")
)

print(f"Found {len(structures)} structures")

if not structures:

    raise FileNotFoundError(
        f"No XYZ files found in: {TRAJ_DIR.resolve()}"
    )


# ============================================================
# Global CNA signature order
# ============================================================
#
# IMPORTANT:
#
# The order is initialized from the first structure in which
# signatures are encountered.
#
# Once a signature has a position, that position is preserved
# for all subsequent structures.
#
# A signature that is not present in a structure is simply
# omitted from that structure's histogram.

# ============================================================

CNA_SIGNATURE_ORDER = []


# ============================================================
# Global aGCN colour scale
# ============================================================
# These are the SAME cmap and norm used by
# add_agcn_snapshot().
# Do not redefine them inside the loop.
# ============================================================

cmap = plt.get_cmap("rainbow")

norm = Normalize(
    vmin=COLOR_SCALE_MIN,
    vmax=COLOR_SCALE_MAX,
    clip=True,
)


pdf = PdfPages(OUTPUT_PDF) if SAVE_PDF else None


try:

    for struct_path in structures:

        struct_name = struct_path.name

        print(
            f"\nProcessing: {struct_name}"
        )


        # ====================================================
        # Read structure
        # ====================================================

        el, coords = read_xyz(
            str(struct_path)
        )

        n_atoms = len(el)

        if n_atoms == 0:

            print(
                "Skipping empty structure"
            )

            continue


        # ====================================================
        # Shape descriptors
        # ====================================================

        asph, acil, rsa = gyr_desc(
            coords
        )

        ar = aspect_ratio(
            coords
        )

        rgyr = gyr_rad(
            coords
        )

        ecc = eccentricity(
            coords
        )

        branch = branch_factor(
            coords,
            npoints=13,
        )


        # ====================================================
        # PDDF
        # ====================================================

        pddf_bins, pddf_counts = (
            pddf_calculator(
                coords,
                bin_width=MY_BIN_SIZE_AA,
                use_lattice_units=False,
            )
        )

        pddf_bins = np.asarray(
            pddf_bins
        )

        pddf_counts = np.asarray(
            pddf_counts
        )

        n_pddf_pairs = np.sum(
            pddf_counts
        )

        if n_pddf_pairs > 0:

            pddf_percentages = (
                100.0
                * pddf_counts
                / n_pddf_pairs
            )

        else:

            pddf_percentages = (
                np.zeros_like(
                    pddf_counts,
                    dtype=float,
                )
            )

        max_pddf = (
            float(
                np.max(pddf_bins)
            )
            if pddf_bins.size
            else np.nan
        )


        # ====================================================
        # RDF
        # ====================================================

        rdf_bins, rdf_counts = (
            com_rdf_calculator(
                coords,
                bin_width=MY_BIN_SIZE_AA,
                elements=el,
            )
        )

        rdf_bins = np.asarray(
            rdf_bins
        )

        rdf_counts = np.asarray(
            rdf_counts
        )


        # ====================================================
        # CNA
        # ====================================================

        n_pairs, signatures = (
            cna.calculate_cna(
                coords,
                CUTOFF,
            )
        )


        if n_pairs > 0:

            sig_count = Counter(
                map(tuple, signatures)
            )



            for signature in sig_count.keys():

                signature = tuple(
                    int(x)
                    for x in signature
                )

                if signature not in CNA_SIGNATURE_ORDER:

                    CNA_SIGNATURE_ORDER.append(
                        signature
                    )


            ordered_signatures = [
                signature
                for signature in CNA_SIGNATURE_ORDER
                if signature in sig_count
            ]


            cna_labels = np.array(
                [
                    f"({int(r)},{int(s)},{int(t)})"
                    for r, s, t in ordered_signatures
                ],
                dtype=object,
            )

            cna_percentages = np.array(
                [
                    100.0
                    * sig_count[signature]
                    / n_pairs
                    for signature
                    in ordered_signatures
                ],
                dtype=float,
            )

            mask = (
                cna_percentages > 0.0
            )

            cna_labels = (
                cna_labels[mask]
            )

            cna_percentages = (
                cna_percentages[mask]
            )

        else:

            cna_labels = np.array(
                [],
                dtype=object,
            )

            cna_percentages = np.array(
                [],
                dtype=float,
            )


        # ====================================================
        # CNAP
        # ====================================================

        cnas_atom = cna.cna_peratom(
            coords,
            CUTOFF,
        )

        (
            n_total,
            n_unique,
            pattern_counter,
        ) = count_cna_patterns(
            cnas_atom
        )

        n_singletons = sum(
            count == 1
            for count
            in pattern_counter.values()
        )

        unique_patterns = (
            pattern_counter.most_common()
        )

        displayed_patterns = (
            unique_patterns[
                :MAX_CNAPS_TO_DISPLAY
            ]
        )


        # ----------------------------------------------------
        # Count [12,(5,5,5)] patterns
        # ----------------------------------------------------

        target_signature = (
            5,
            5,
            5,
        )

        n_12_555 = 0

        for sigs, counts in cnas_atom:

            if len(sigs) == 1:

                signature = tuple(
                    sigs[0].astype(int)
                )

                if (
                    signature
                    == target_signature
                    and counts[0] == 12
                ):

                    n_12_555 += 1


        # ====================================================
        # aGCN
        # ====================================================

        sites, agcns = (
            agcn_calculator(
                coords,
                CUTOFF,
            )
        )

        agcns = np.asarray(
            agcns,
            dtype=float,
        )


        # ----------------------------------------------------
        # aGCN histogram
        # ----------------------------------------------------

        bin_edges = np.arange(
            AGCN_MIN,
            AGCN_MAX + AGCN_BIN_WIDTH,
            AGCN_BIN_WIDTH,
        )

        agcn_occurrences, _ = (
            np.histogram(
                agcns,
                bins=bin_edges,
            )
        )

        if len(agcns) > 0:

            agcn_percentages = (
                100.0
                * agcn_occurrences
                / len(agcns)
            )

        else:

            agcn_percentages = (
                np.zeros_like(
                    agcn_occurrences,
                    dtype=float,
                )
            )

        bin_centers = (
            bin_edges[:-1]
            + bin_edges[1:]
        ) / 2.0

        bar_colors = cmap(
            norm(bin_centers)
        )


        # ====================================================
        # Structure snapshot paths
        # ====================================================

        structure_png_path = (
            SNAPSHOT_DIR
            / f"{struct_path.stem}"
              f"{SNAPSHOT_SUFFIX}"
        )


        # ====================================================
        # aGCN snapshot path
        # ====================================================

        agcn_png_path = (
            SNAPSHOT_DIR
            / f"{struct_path.stem}_agcn.png"
        )


        # ====================================================
        # Figure
        # ====================================================

        fig = plt.figure(
            figsize=FIGURE_SIZE,
            facecolor="white",
        )

        fig.subplots_adjust(
            left=0.035,
            right=0.992,
            bottom=0.125,
            top=0.975,
            wspace=0.24,
            hspace=0.52,
        )

        gs = GridSpec(
            2,
            3,
            figure=fig,
            width_ratios=[
                0.33,
                1.40,
                1.30,
            ],
            height_ratios=[
                1.0,
                1.0,
            ],
        )


        # ====================================================
        # Shape table
        # ====================================================

        ax_shape = fig.add_subplot(
            gs[:, 0]
        )

        shape_rows = [

            (
                r"$N$",
                f"{n_atoms}",
            ),

            (
                r"$R_g$ [\AA]",
                f"{rgyr:.1f}",
            ),

            (
                r"$\kappa^2$",
                f"{rsa:.3f}",
            ),

            (
                r"$\varepsilon$",
                f"{ecc:.1f}",
            ),

            (
                r"$f$",
                f"{branch:.1f}",
            ),

        ]

        draw_shape_table(
            ax_shape,
            shape_rows,
        )


        # ====================================================
        # PDDF
        # ====================================================

        ax_pddf = fig.add_subplot(
            gs[0, 1]
        )

        ax_pddf.bar(
            pddf_bins,
            pddf_percentages,
            width=MY_BIN_SIZE_AA,
            align="center",
            color=MAIN_COLOR,
            edgecolor=MAIN_COLOR,
            linewidth=0.18,
            zorder=2,
        )

        ax_pddf.set_xlabel(
            r"Pair distance [$\AA$]",
            labelpad=2.5,
        )

        ax_pddf.set_ylabel(
            "Occurrence [%]",
            labelpad=3.0,
        )

        if np.isfinite(max_pddf):

            ax_pddf.set_xlim(
                0,
                max_pddf * 1.06,
            )

        ax_pddf.set_ylim(
            0,
            padded_ylim(
                pddf_percentages,
                pad=1.18,
            ),
        )

        clean_axis(
            ax_pddf
        )

        add_panel_label(
            ax_pddf,
            "(a)",
        )


        # ----------------------------------------------------
        # Existing structure snapshot:
        #
        # use it if present.
        # Otherwise generate a new ASE snapshot.
        # ----------------------------------------------------

        if structure_png_path.exists():

            add_snapshot(
                ax_pddf,
                structure_png_path,
                PDDF_SNAPSHOT_CENTER_X,
                PDDF_SNAPSHOT_CENTER_Y,
                SNAPSHOT_WIDTH,
                SNAPSHOT_HEIGHT,
            )

        else:

            add_structure_snapshot(
                ax_pddf,
                el,
                coords,
                PDDF_SNAPSHOT_CENTER_X,
                PDDF_SNAPSHOT_CENTER_Y,
                SNAPSHOT_WIDTH,
                SNAPSHOT_HEIGHT,
            )


        # ====================================================
        # aGCN
        # ====================================================

        ax_agcn = fig.add_subplot(
            gs[0, 2]
        )


        # ----------------------------------------------------
        # Histogram bars
        # ----------------------------------------------------

        ax_agcn.bar(
            bin_centers,
            agcn_percentages,
            width=AGCN_BIN_WIDTH * 0.88,
            align="center",
            color=bar_colors,
            edgecolor=EDGE_COLOR,
            linewidth=0.30,
            zorder=2,
        )


        # ----------------------------------------------------
        # Common colourbar
        # ----------------------------------------------------

        sm = plt.cm.ScalarMappable(
            cmap=cmap,
            norm=norm,
        )

        sm.set_array([])

        cbar = fig.colorbar(
            sm,
            ax=ax_agcn,
            pad=0.02,
            fraction=0.05,
        )

        cbar.set_label(
            "aGCN"
        )

        cbar.set_ticks(
            np.arange(
                COLOR_SCALE_MIN,
                COLOR_SCALE_MAX + 0.01,
                2.0,
            )
        )


        # ----------------------------------------------------
        # aGCN axes
        # ----------------------------------------------------

        ax_agcn.set_xlabel(
            "aGCN",
            labelpad=2.5,
        )

        ax_agcn.set_ylabel(
            "Occurrence [%]",
            labelpad=3.0,
        )

        ax_agcn.set_xlim(
            AGCN_MIN,
            AGCN_MAX,
        )

        ax_agcn.set_xticks(
            np.arange(
                AGCN_MIN,
                AGCN_MAX + 0.01,
                2.0,
            )
        )

        positive_agcn = (
            agcn_percentages[
                agcn_percentages > 0.0
            ]
        )

        ax_agcn.set_ylim(
            0,
            padded_ylim(
                positive_agcn,
                pad=1.15,
            ),
        )

        clean_axis(
            ax_agcn
        )

        add_panel_label(
            ax_agcn,
            "(b)",
        )


        if agcn_png_path.exists():

            add_snapshot(
                ax_agcn,
                agcn_png_path,
                AGCN_SNAPSHOT_CENTER_X,
                AGCN_SNAPSHOT_CENTER_Y,
                SNAPSHOT_WIDTH,
                SNAPSHOT_HEIGHT,
            )

        else:

            add_agcn_snapshot(
                ax_agcn,
                el,
                coords,
                agcns,
                AGCN_SNAPSHOT_CENTER_X,
                AGCN_SNAPSHOT_CENTER_Y,
                SNAPSHOT_WIDTH,
                SNAPSHOT_HEIGHT,
            )


        # ====================================================
        # CNA
        # ====================================================

        ax_cna = fig.add_subplot(
            gs[1, 1]
        )

        x_positions = np.arange(
            len(cna_labels)
        )


        # ----------------------------------------------------
        # Target CNA
        # ----------------------------------------------------

        cna_target_mask = np.array(
            [
                label == TARGET_CNA_LABEL
                for label in cna_labels
            ],
            dtype=bool,
        )

        cna_left_percentages = (
            cna_percentages.copy()
        )

        cna_left_percentages[
            cna_target_mask
        ] = 0.0

        cna_target_percentage = (
            float(
                cna_percentages[
                    cna_target_mask
                ][0]
            )
            if np.any(cna_target_mask)
            else 0.0
        )


        # ----------------------------------------------------
        # CNA bars
        # ----------------------------------------------------

        bars_cna = ax_cna.bar(
            x_positions,
            cna_left_percentages,
            width=0.70,
            color=CNA_COLOR,
            edgecolor=EDGE_COLOR,
            linewidth=0.35,
            zorder=2,
        )

        ax_cna.set_xlabel(
            "CNA signature",
            labelpad=2.5,
        )

        ax_cna.set_ylabel(
            "Occurrence [%]",
            labelpad=3.0,
        )

        ax_cna.set_xticks(
            x_positions
        )

        ax_cna.set_xticklabels(
            cna_labels,
            rotation=43,
            ha="right",
            rotation_mode="anchor",
            fontsize=7.7,
        )

        ax_cna.set_ylim(
            0,
            padded_ylim(
                cna_left_percentages[
                    cna_left_percentages > 0.0
                ],
                pad=1.15,
            ),
        )

        clean_axis(
            ax_cna
        )

        add_panel_label(
            ax_cna,
            "(c)",
        )

        if cna_percentages.size == 0:

            ax_cna.text(
                0.50,
                0.50,
                "No CNA pairs",
                transform=ax_cna.transAxes,
                ha="center",
                va="center",
                fontsize=9.0,
                color="0.45",
            )


        # ====================================================
        # CNAP
        # ====================================================

        ax_cnap = fig.add_subplot(
            gs[1, 2]
        )

        ax_cnap.set_xlim(
            0,
            1,
        )

        ax_cnap.set_ylim(
            0,
            1,
        )

        ax_cnap.axis(
            "off"
        )

        add_panel_label(
            ax_cnap,
            "(d)",
            x=-0.13,
        )


        summary_labels = [

            r"$\mathcal{N}_{\mathrm{CNAP}}$",

            r"$\mathcal{U}_{\mathrm{CNAP}}$",

            r"$\mathcal{S}_{\mathrm{CNAP}}$",

            r"$\mathcal{N}_{[12,(5,5,5)]}$",

        ]


        summary_values = [

            n_total,

            n_unique,

            n_singletons,

            n_12_555,

        ]


        for x, label, value in zip(

            [
                0.12,
                0.37,
                0.62,
                0.87,
            ],

            summary_labels,

            summary_values,

        ):

            ax_cnap.text(
                x,
                0.940,
                label,
                transform=ax_cnap.transAxes,
                fontsize=7.6,
                ha="center",
                va="center",
                color=TEXT_COLOR,
            )

            ax_cnap.text(
                x,
                0.855,
                f"{value}",
                transform=ax_cnap.transAxes,
                fontsize=8.7,
                fontweight="semibold",
                ha="center",
                va="center",
                color=TEXT_COLOR,
            )


        ax_cnap.plot(
            [0.015, 0.985],
            [0.790, 0.790],
            transform=ax_cnap.transAxes,
            color=LIGHT_GREY,
            linewidth=0.75,
        )


        if displayed_patterns:

            header = (

                rf"$\mathrm{{CNAP}}$ "
                rf"(top {MAX_CNAPS_TO_DISPLAY})"

                if n_unique > MAX_CNAPS_TO_DISPLAY

                else r"$\mathrm{CNAP}$"

            )


            ax_cnap.text(
                CNAP_TEXT_X,
                0.735,
                header,
                transform=ax_cnap.transAxes,
                fontsize=7.4,
                fontweight="bold",
                ha="left",
                va="center",
                color="0.30",
            )

            ax_cnap.text(
                CNAP_NATOMS_X,
                0.735,
                r"$n_{\mathrm{atoms}}$",
                transform=ax_cnap.transAxes,
                fontsize=7.4,
                fontweight="bold",
                ha="right",
                va="center",
                color="0.30",
            )


            ax_cnap.plot(
                [0.015, 0.985],
                [0.690, 0.690],
                transform=ax_cnap.transAxes,
                color=VERY_LIGHT_GREY,
                linewidth=0.70,
            )


            formatted_patterns = [

                format_pattern_lines(
                    pattern,
                    max_chars_per_line=CNAP_MAX_CHARS_PER_LINE,
                )

                for pattern, count
                in displayed_patterns

            ]


            n_lines_per_pattern = [

                text.count("\n") + 1

                for text
                in formatted_patterns

            ]


            available_height = (
                CNAP_TABLE_TOP
                - CNAP_TABLE_BOTTOM
            )


            total_text_units = sum(
                n_lines_per_pattern
            )


            total_gap_units = (
                CNAP_ENTRY_GAP_UNITS
                * max(
                    len(displayed_patterns) - 1,
                    0,
                )
            )


            unit_height = (
                available_height
                / max(
                    total_text_units
                    + total_gap_units,
                    1,
                )
            )


            current_y = (
                CNAP_TABLE_TOP
            )


            for (
                i,
                ((pattern, count), text, n_lines),
            ) in enumerate(

                zip(
                    displayed_patterns,
                    formatted_patterns,
                    n_lines_per_pattern,
                ),

                start=1,

            ):

                text_height = (
                    n_lines
                    * unit_height
                )

                row_center = (
                    current_y
                    - text_height / 2.0
                )


                background_height = (
                    text_height
                    + CNAP_ENTRY_GAP_UNITS
                    * unit_height
                    * 0.72
                )


                if i % 2 == 0:

                    rect = Rectangle(

                        (
                            0.010,
                            row_center
                            - background_height / 2.0,
                        ),

                        0.98,
                        background_height,

                        transform=ax_cnap.transAxes,

                        facecolor="0.97",
                        edgecolor="none",

                        zorder=0,

                    )

                    ax_cnap.add_patch(
                        rect
                    )


                ax_cnap.text(
                    CNAP_NUMBER_X,
                    row_center,
                    f"{i}.",
                    transform=ax_cnap.transAxes,
                    fontsize=CNAP_FONT_SIZE,
                    ha="left",
                    va="center",
                    color="0.45",
                    zorder=2,
                )


                ax_cnap.text(
                    CNAP_TEXT_X,
                    row_center,
                    text,
                    transform=ax_cnap.transAxes,
                    fontsize=CNAP_FONT_SIZE,
                    family="monospace",
                    ha="left",
                    va="center",
                    multialignment="left",
                    linespacing=CNAP_LINE_SPACING,
                    color=TEXT_COLOR,
                    zorder=2,
                )


                ax_cnap.text(
                    CNAP_NATOMS_X,
                    row_center,
                    f"{count}",
                    transform=ax_cnap.transAxes,
                    fontsize=CNAP_FONT_SIZE,
                    ha="right",
                    va="center",
                    color=TEXT_COLOR,
                    zorder=4,
                    bbox=dict(
                        facecolor="white",
                        edgecolor="none",
                        pad=0.25,
                        alpha=0.96,
                    ),
                )


                current_y -= (
                    text_height
                )


                if i < len(
                    displayed_patterns
                ):

                    current_y -= (
                        CNAP_ENTRY_GAP_UNITS
                        * unit_height
                    )

        else:

            ax_cnap.text(
                0.50,
                0.40,
                "No CNAPs",
                transform=ax_cnap.transAxes,
                fontsize=8.5,
                ha="center",
                va="center",
                color="0.45",
            )


        # ====================================================
        # Terminal summary
        # ====================================================

        print(
            "\n--- Shape descriptors ---"
        )

        print(
            f"Radius of gyration [Å]:    "
            f"{rgyr:.6f}"
        )

        print(
            f"Asphericity:               "
            f"{asph:.6e}"
        )

        print(
            f"Acylindricity:             "
            f"{acil:.6e}"
        )

        print(
            f"Relative shape anisotropy: "
            f"{rsa:.6e}"
        )

        print(
            f"Aspect ratio:              "
            f"{ar:.6e}"
        )

        print(
            f"Eccentricity:              "
            f"{ecc:.6f}"
        )

        print(
            f"Branch factor:             "
            f"{branch:.6f}"
        )


        print(
            "\n--- CNAP summary ---"
        )

        print(
            f"N_CNAP = {n_total}"
        )

        print(
            f"U_CNAP = {n_unique}"
        )

        print(
            f"S_CNAP = {n_singletons}"
        )

        print(
            f"N_[12,(5,5,5)] = {n_12_555}"
        )


        # ====================================================
        # Save PNG
        # ====================================================

        output_png = (
            OUTPUT_DIR
            / f"ID_2D_{struct_path.stem}.png"
        )

        fig.savefig(
            output_png,
            dpi=OUTPUT_DPI,
            bbox_inches="tight",
            pad_inches=0.025,
            facecolor="white",
        )


        # ====================================================
        # Save PDF
        # ====================================================

        if pdf is not None:

            pdf.savefig(
                fig,
                dpi=OUTPUT_DPI,
                bbox_inches="tight",
                pad_inches=0.025,
                facecolor="white",
            )


        plt.close(
            fig
        )

        print(
            f"Saved: {output_png}"
        )


finally:

    if pdf is not None:

        pdf.close()


print(
    "\nAll reports generated successfully."
)

print(
    "\nGlobal CNA signature order:"
)

for i, signature in enumerate(
    CNA_SIGNATURE_ORDER,
    start=1,
):

    print(
        f"  {i}. {signature}"
    )

Found 5 structures

Processing: Au13_ih.xyz

--- Shape descriptors ---
Radius of gyration [Å]:    2.771810
Asphericity:               0.000000e+00
Acylindricity:             0.000000e+00
Relative shape anisotropy: -1.110223e-16
Aspect ratio:              1.000000e+00
Eccentricity:              1.000000
Branch factor:             0.334408

--- CNAP summary ---
N_CNAP = 13
U_CNAP = 2
S_CNAP = 1
N_[12,(5,5,5)] = 1
Saved: reports/ID_2D_Au13_ih.png

Processing: Cu247_Dh.xyz

--- Shape descriptors ---
Radius of gyration [Å]:    11.869300
Asphericity:               1.237751e+02
Acylindricity:             6.773561e-09
Relative shape anisotropy: 7.719088e-01
Aspect ratio:              8.435856e-02
Eccentricity:              0.195832
Branch factor:             2.009060

--- CNAP summary ---
N_CNAP = 247
U_CNAP = 8
S_CNAP = 0
N_[12,(5,5,5)] = 0
Saved: reports/ID_2D_Cu247_Dh.png

Processing: Cu309_Ih.xyz

--- Shape descriptors ---
Radius of gyration [Å]:    7.551884
Asphericity:               7.10